# Deploy and Run Hunyuan3D-2.1 on Modal

This notebook deploys the Tencent Hunyuan3D-2.1 image-to-3D model to Modal and runs GLB generation inference on an H100 GPU.


In [ ]:
# Deploy the Hunyuan3D-2.1 model to Modal
!uv run modal deploy ../llm-hosting/hunyuan3d-2.py


## Generate a 3D Model from an Image

The deployed Modal class accepts image bytes, generates a shape mesh with Hunyuan3D-2.1, returns the verified shape GLB bytes. The notebook validates the GLB header before saving the output.


In [ ]:
from pathlib import Path
import modal
import urllib.request

APP_NAME = "hunyuan3d-2"
CLASS_NAME = "Hunyuan3DModel"
TEXTURE = False
REMOVE_BACKGROUND = True

# Connect to the deployed Modal app class
Hunyuan3DModel = modal.Cls.from_name(APP_NAME, CLASS_NAME)
model = Hunyuan3DModel()

# Use the local sample if available, otherwise download the official demo image
input_image_path = Path("../llm-hosting/demo.png")
if not input_image_path.exists():
    demo_url = "https://raw.githubusercontent.com/Tencent-Hunyuan/Hunyuan3D-2.1/main/assets/demo.png"
    input_image_path = Path("demo.png")
    print(f"Downloading demo image from {demo_url}...")
    urllib.request.urlretrieve(demo_url, input_image_path)

image_bytes = input_image_path.read_bytes()

print("Sending inference request to Hunyuan3D-2.1 on Modal...")
glb_bytes = model.generate.remote(
    image_bytes,
    texture=TEXTURE,
    remove_background=REMOVE_BACKGROUND,
)

if len(glb_bytes) < 1_000:
    raise ValueError(f"Generated GLB is too small: {len(glb_bytes)} bytes")
if glb_bytes[:4] != b"glTF":
    raise ValueError("Generated output is not a binary GLB file")

output_glb_path = Path("hunyuan3d-2.1-output.glb")
output_glb_path.write_bytes(glb_bytes)

print(f"3D model saved to: {output_glb_path} ({len(glb_bytes)} bytes)")


### Visualizing the 3D model

Open `hunyuan3d-2.1-output.glb` in a standard 3D viewer such as Windows 3D Viewer or https://gltf-viewer.donmccurdy.com/.
